In [3]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme('paper', font_scale=0.8)

import geneinfo.information as gi

plt.rcParams['figure.facecolor'], plt.rcParams['axes.facecolor'] = '#1F1F1F', '#1F1F1F' 

%config InlineBackend.figure_formats = ['svg'] 

In [4]:
common_miss = pd.read_csv('../results/common_missense_human_gnomad.csv')
common_miss

,species,gene_id,gene_name,transcript_id,variant_id,chromosome,genomic_position,ref_allele,alt_allele,aa_position,ref_aa,alt_aa,allele_frequency,hgvsp,hgvsc,consequence,source
0,human,ENSG00000169084,DHRSX,ENST00000334651,X-2221145-C-T,X,2221145,C,T,297,Glu,Lys,0.333643,p.Glu297Lys,c.889G>A,missense_variant,gnomAD_v4
1,human,ENSG00000169084,DHRSX,ENST00000334651,X-2221159-T-C,X,2221159,T,C,292,His,Arg,0.868850,p.His292Arg,c.875A>G,missense_variant,gnomAD_v4
2,human,ENSG00000169084,DHRSX,ENST00000334651,X-2243088-C-G,X,2243088,C,G,247,Val,Leu,0.709305,p.Val247Leu,c.739G>C,missense_variant,gnomAD_v4
3,human,ENSG00000205755,CRLF2,ENST00000400841,X-1193297-T-C,X,1193297,T,C,258,Lys,Arg,0.502598,p.Lys258Arg,c.773A>G,missense_variant,gnomAD_v4
4,human,ENSG00000196433,ASMT,ENST00000381241,X-1632709-T-C,X,1632709,T,C,190,Trp,Arg,0.444030,p.Trp190Arg,c.568T>C,missense_variant,gnomAD_v4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17204,human,ENSG00000114473,IQCG,ENST00000265239,3-197932289-C-G,3,197932289,C,G,177,Asp,His,0.248446,p.Asp177His,c.529G>C,missense_variant,gnomAD_v4
17205,human,ENSG00000114473,IQCG,ENST00000265239,3-197938728-G-T,3,197938728,G,T,112,Ala,Asp,0.248504,p.Ala112Asp,c.335C>A,missense_variant,gnomAD_v4
17206,human,ENSG00000061938,TNK2,ENST00000672887,3-195868079-G-A,3,195868079,G,A,740,Pro,Leu,0.213131,p.Pro740Leu,c.2219C>T,missense_variant,gnomAD_v4
17207,human,ENSG00000122068,FYTTD1,ENST00000241502,3-197768463-G-A,3,197768463,G,A,87,Arg,His,0.835541,p.Arg87His,c.260G>A,missense_variant,gnomAD_v4


In [7]:
all_lof = pd.read_parquet("../results/gnomad_lof/gnomad_lof_all_genes_20251206_003756.parquet")
all_lof


,gene_name,gene_id,transcript_id,variant_id,chrom,position,ref,alt,mutation_type,consequence,...,af_total,af_afr,af_amr,af_asj,af_eas,af_fin,af_nfe,af_oth,af_sas,cds_relative_position
0,A1BG,ENSG00000121410,ENST00000263100,19-58858397-T-C,19,58858397,T,C,Splice acceptor,splice_acceptor_variant,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000009,0.000000,0.000000,Unknown
1,A1BG,ENSG00000121410,ENST00000263100,19-58858718-C-T,19,58858718,C,T,Splice donor,splice_donor_variant,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000033,Unknown
2,A1BG,ENSG00000121410,ENST00000263100,19-58858752-C-A,19,58858752,C,A,In-frame stop codon,stop_gained,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000033,Unknown
3,A1BG,ENSG00000121410,ENST00000263100,19-58858774-C-T,19,58858774,C,T,In-frame stop codon,stop_gained,...,0.000004,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000168,0.000000,Unknown
4,A1BG,ENSG00000121410,ENST00000263100,19-58858780-G-GTA,19,58858780,G,GTA,Frameshift,frameshift_variant,...,0.000008,0.000000,0.000000,0.000000,0.000113,0.0,0.000000,0.000000,0.000000,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715938,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185236-G-GC,19,14185236,G,GC,Frameshift,frameshift_variant,...,0.000081,0.000000,0.000000,0.000121,0.000096,0.0,0.000094,0.000240,0.000134,Unknown
715939,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185276-G-A,19,14185276,G,A,Splice donor,splice_donor_variant,...,0.000032,0.000115,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown
715940,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185514-CCA-C,19,14185514,CCA,C,Splice acceptor,splice_acceptor_variant,...,0.000007,0.000000,0.000041,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown
715941,hsa-mir-1199,ENSG00000141854,ENST00000269720,19-14185516-A-AGGG,19,14185516,A,AGGG,Splice acceptor,splice_acceptor_variant,...,0.000007,0.000000,0.000041,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,Unknown


In [32]:
common_lof = common_miss.merge(all_lof, 
                               left_on=['gene_name', 'variant_id'], 
                               right_on=['gene_name', 'variant_id'], how='inner',
                                # suffixes=('_com', '_lof')
                                )
common_lof

,species,gene_id_x,gene_name,transcript_id_x,variant_id,chromosome,genomic_position,ref_allele,alt_allele,aa_position,...,af_total,af_afr,af_amr,af_asj,af_eas,af_fin,af_nfe,af_oth,af_sas,cds_relative_position


In [ ]:
df = pd.read_parquet("../data/alpha_missense_hg38.parquet")
regex = re.compile(r'(.)(\d+)(.)')
ref, pos, alt = zip(*[regex.match(x).groups() for x in df.protein_variant])
df['ref_aa'] = ref
df['pos'] = list(map(int, pos))
df['alt_aa'] = alt
df

In [ ]:
uniprot_ids = df.uniprot_id.unique()
uniprot_ids

In [ ]:
uniprot2hgcn = {}
for x in tqdm(df.uniprot_id.unique()):
    try:
        uniprot2hgcn[x] = gi.hgcn_symbol(x)
    except gi.NotFound:
        uniprot2hgcn[x] = None

  0%|          | 0/20516 [00:00<?, ?it/s]

NotFound: 

In [ ]:
common_miss.join(df, on=['uniprot_id', 'pos', 'ref_aa', 'alt_aa'], how='left')

In [ ]:
uniprot_id = 'Q8NHH1'
df = pd.read_parquet("../data/alpha_missense_hg38.parquet", filters=[("uniprot_id", "==", f"{uniprot_id}")])
regex.match(r'(.)(\d+)(.)').groups() df.protein_variant
df

,uniprot_id,protein_variant,am_pathogenicity,am_class
0,Q8NHH1,M1A,0.3139,benign
1,Q8NHH1,M1C,0.4108,ambiguous
2,Q8NHH1,M1D,0.8912,pathogenic
3,Q8NHH1,M1E,0.7156,pathogenic
4,Q8NHH1,M1F,0.2341,benign
...,...,...,...,...
15195,Q8NHH1,S800R,0.2823,benign
15196,Q8NHH1,S800T,0.0887,benign
15197,Q8NHH1,S800V,0.1524,benign
15198,Q8NHH1,S800W,0.1835,benign


In [ ]:
pd.read_csv('../results/gnomad_lof/gnomad_lof_all_genes_20251206_003756.csv')